# Gas Giant Generator

This notebook uses the `SphereCamera` class from `render/camera.py` to render gas giants with rings.

In [ ]:
# Environment Setup
import os
import sys
import subprocess

# 1. Handle Colab/Local Filesystem and Pull Latest
IS_COLAB = 'google.colab' in sys.modules
REPO_URL = "https://github.com/pmcray/cosmic_engine.git"
PROJECT_NAME = "cosmic_engine"

def setup_environment():
    if IS_COLAB:
        print("Running on Google Colab.")
        if not os.path.exists(f"/content/{PROJECT_NAME}"):
            print(f"Cloning {PROJECT_NAME} repository...")
            subprocess.run(["git", "clone", REPO_URL], check=True)
        
        # Change to project root & pull latest changes
        os.chdir(f"/content/{PROJECT_NAME}")
        print("Pulling latest updates from GitHub...")
        subprocess.run(["git", "pull"], check=True)
        sys.path.append(os.getcwd())
    else:
        # Local setup: move up from /notebooks to the project root
        current_dir = os.getcwd()
        if os.path.basename(current_dir) == "notebooks":
            root_dir = os.path.abspath(os.path.join(current_dir, ".."))
        else:
            root_dir = current_dir
            
        if root_dir not in sys.path:
            sys.path.append(root_dir)
        os.chdir(root_dir)
        print(f"Local root added to sys.path: {root_dir}")
        print("Executing git pull to ensure latest changes... (Locally)")
        try:
            subprocess.run(["git", "pull"], check=True)
        except Exception as e:
            print(f"Git pull warnings (ignoring): {e}")

setup_environment()

# 2. Dependency Management
try:
    import taichi as ti
    import numpy as np
    from PIL import Image as PILImage
    print("Dependencies already installed.")
except ImportError:
    print("Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "taichi", "numpy", "Pillow"], check=True)
    import taichi as ti
    import numpy as np
    from PIL import Image as PILImage

print(f"Current working directory: {os.getcwd()}")
print(f"Taichi version: {ti.__version__}")

In [ ]:
import random
import taichi as ti
import numpy as np
from render.camera import SphereCamera
from physics.fluid_solver import FluidEngine
from IPython.display import display, Image
import PIL.Image

# ---------------------------------------------------------
# SIMULATION & RENDER OPTIONS (EASY TOGGLES)
# ---------------------------------------------------------
# Planet Options: "jupiter", "saturn", "uranus", "venus", "hot_jupiter"
PLANET_TYPE = "jupiter"

# Note on "Refined Scale" and "Longer Simulation":
# 1. Higher RES: Yields massive improvements in micro-turbulent details. 
#    A refined grid (e.g. 2048x2048) creates far sharper and more photorealistic atmospheric structures.
# 2. Higher SIM_FRAMES: Lets chaotic fluid dynamics (like storm mergers and shear instability) develop 
#    deeply over time.

RES = 1024                # Up this to 2048 for ultimate ultra-photorealism (requires high VRAM!)
SIM_FRAMES = 1200         # Higher = more mature storm mechanics and interacting vortex streets

# Render Settings
HAS_RINGS = 0             # 1 for rings, 0 for no rings (best for Saturn/Uranus)
SAMPLES = 4               # AA Quality: Increase for smoother edges, at cost of render time

# Dynamic Juno/Flyby Camera Angles
RANDOMIZE_CAMERA = True   # Automatically spin and tilt the camera for dramatically varied flyby shots
# ---------------------------------------------------------

# 1. Initialize Engines
# Provide randomized seed to ensure identical options don't produce the identical image twice
ti.init(arch=ti.gpu, random_seed=int(random.random() * 1000000))

fluid = FluidEngine(res=RES, planet_type=PLANET_TYPE)
camera = SphereCamera(fluid_res=RES, render_res=RES, has_rings=HAS_RINGS, samples=SAMPLES)

if RANDOMIZE_CAMERA:
    camera.cam_tilt = random.uniform(-1.0, 1.0)
    camera.cam_pan = random.uniform(-0.4, 0.4) 
    camera.cam_roll = random.uniform(-0.2, 0.2) 
    
    light_x = random.uniform(0.3, 0.9)
    light_y = random.uniform(-0.6, 0.6)
    light_z = -1.0
else:
    light_x, light_y, light_z = 0.5, 0.3, -1.0

# 2. Run Fluid Simulation to establish atmosphere
print(f"Simulating {PLANET_TYPE} atmosphere for {SIM_FRAMES} frames @ {RES}x{RES} scale...")
for f in range(SIM_FRAMES):
    fluid.step(f)
    if f % 100 == 0:
        print(f"  Frame {f}/{SIM_FRAMES}")

# 3. Render 3D Gas Giant
if RANDOMIZE_CAMERA:
    print(f"\nRendering final output (Tilt: {camera.cam_tilt:.2f}, Pan: {camera.cam_pan:.2f})...")
else:
    print(f"\nRendering final output...")
    
camera.render_gas_giant(fluid.dye, light_x, light_y, light_z)

img = camera.get_image_data()
output_img = PIL.Image.fromarray((img * 255).astype(np.uint8))
display(output_img)
print("Done!")